In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

from llm import extract
from prompts import TECHNIQUES

with open(ROOT / "data" / "examples.json", encoding="utf-8") as f:
    examples = json.load(f)

print(len(examples), "examples |", list(TECHNIQUES))

24 examples | ['zero_shot', 'few_shot', 'role', 'chain_of_thought']


In [2]:
RESULTS_PATH = ROOT / "data" / "results_extraction.json"


def run_all():
    results = []
    for technique in TECHNIQUES:
        for ex in examples:
            start = time.perf_counter()
            try:
                prediction, retries = extract(ex["input"], technique=technique)
                predicted, error = prediction.model_dump(), None
            except Exception as exc:
                predicted, retries, error = None, None, f"{type(exc).__name__}: {exc}"
            seconds = round(time.perf_counter() - start, 2)
            results.append({
                "technique": technique,
                "id": ex["id"],
                "predicted": predicted,
                "retries": retries,
                "seconds": seconds,
                "error": error,
            })
            print(f"{technique:17} {ex['id']}  {seconds:5.1f}s  {'ERROR' if error else 'ok'}")
    return results


if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    print("Loaded saved results:", len(results))
else:
    results = run_all()
    RESULTS_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
    print("Saved", len(results), "results")

Loaded saved results: 96


In [3]:
FIELDS = ["job_title", "seniority", "required_skills", "nice_to_have_skills",
          "salary_min", "salary_max", "remote"]


def normalise(field, value):
    if field in ("required_skills", "nice_to_have_skills"):
        return {skill.strip().lower() for skill in value}
    if field == "job_title":
        return value.strip().lower()
    return value


expected_by_id = {ex["id"]: ex["expected"] for ex in examples}

rows = []
for r in results:
    expected = expected_by_id[r["id"]]
    for field in FIELDS:
        if r["predicted"] is None:
            correct = False
        else:
            correct = normalise(field, r["predicted"][field]) == normalise(field, expected[field])
        rows.append({"technique": r["technique"], "id": r["id"], "field": field, "correct": correct})

scores = pd.DataFrame(rows)
scores.head(10)

,technique,id,field,correct
0,zero_shot,ex01,job_title,True
1,zero_shot,ex01,seniority,True
2,zero_shot,ex01,required_skills,True
3,zero_shot,ex01,nice_to_have_skills,True
4,zero_shot,ex01,salary_min,True
5,zero_shot,ex01,salary_max,True
6,zero_shot,ex01,remote,True
7,zero_shot,ex02,job_title,True
8,zero_shot,ex02,seniority,True
9,zero_shot,ex02,required_skills,True


In [4]:
table = scores.pivot_table(index="technique", columns="field", values="correct", aggfunc="mean") * 100
table = table[FIELDS]
table["overall"] = table.mean(axis=1)

per_example = scores.groupby(["technique", "id"])["correct"].all()
table["all_fields_correct"] = per_example.groupby("technique").mean() * 100

runs = pd.DataFrame(results)
table["avg_seconds"] = runs.groupby("technique")["seconds"].mean()
table["retries"] = runs.groupby("technique")["retries"].sum()
table["failures"] = runs.groupby("technique")["error"].count()

table = table.reindex(list(TECHNIQUES)).round(1)
table

field,job_title,seniority,required_skills,nice_to_have_skills,salary_min,salary_max,remote,overall,all_fields_correct,avg_seconds,retries,failures
technique,,,,,,,,,,,,
zero_shot,100.0,100.0,87.5,100.0,100.0,100.0,100.0,98.2,87.5,0.8,0,0
few_shot,100.0,100.0,95.8,100.0,100.0,100.0,100.0,99.4,95.8,0.8,0,0
role,100.0,100.0,87.5,100.0,100.0,100.0,100.0,98.2,87.5,1.0,0,0
chain_of_thought,100.0,100.0,91.7,100.0,100.0,100.0,100.0,98.8,91.7,1.2,0,0


In [5]:
lookup = {(r["technique"], r["id"]): r for r in results}

mistakes = scores[~scores["correct"]].copy()
mistakes["expected"] = [expected_by_id[i][f] for i, f in zip(mistakes["id"], mistakes["field"])]
mistakes["predicted"] = [
    (lookup[(t, i)]["predicted"] or {}).get(f)
    for t, i, f in zip(mistakes["technique"], mistakes["id"], mistakes["field"])
]

pd.set_option("display.max_colwidth", None)
mistakes.sort_values(["id", "field", "technique"])

,technique,id,field,correct,expected,predicted
583,chain_of_thought,ex12,required_skills,False,"[Excel, Workday]","[Excel, Workday, HR reporting]"
415,role,ex12,required_skills,False,"[Excel, Workday]","[Excel, Workday, HR reporting]"
79,zero_shot,ex12,required_skills,False,"[Excel, Workday]","[Excel, Workday, HR reporting]"
457,role,ex18,required_skills,False,"[Collibra, data quality frameworks, GDPR]","[stakeholder management, communication skills, Collibra, data quality frameworks, GDPR]"
121,zero_shot,ex18,required_skills,False,"[Collibra, data quality frameworks, GDPR]","[Collibra, data quality frameworks, GDPR, stakeholder management, communication skills]"
660,chain_of_thought,ex23,required_skills,False,"[PostgreSQL, MySQL, Bash]","[PostgreSQL, MySQL, Bash scripts, backups]"
324,few_shot,ex23,required_skills,False,"[PostgreSQL, MySQL, Bash]","[PostgreSQL, MySQL, Bash, backups]"
492,role,ex23,required_skills,False,"[PostgreSQL, MySQL, Bash]","[PostgreSQL, MySQL, Bash, backups]"
156,zero_shot,ex23,required_skills,False,"[PostgreSQL, MySQL, Bash]","[PostgreSQL, MySQL, Bash, backups]"


In [6]:
print(table.to_markdown())

| technique        |   job_title |   seniority |   required_skills |   nice_to_have_skills |   salary_min |   salary_max |   remote |   overall |   all_fields_correct |   avg_seconds |   retries |   failures |
|:-----------------|------------:|------------:|------------------:|----------------------:|-------------:|-------------:|---------:|----------:|---------------------:|--------------:|----------:|-----------:|
| zero_shot        |         100 |         100 |              87.5 |                   100 |          100 |          100 |      100 |      98.2 |                 87.5 |           0.8 |         0 |          0 |
| few_shot         |         100 |         100 |              95.8 |                   100 |          100 |          100 |      100 |      99.4 |                 95.8 |           0.8 |         0 |          0 |
| role             |         100 |         100 |              87.5 |                   100 |          100 |          100 |      100 |      98.2 |               